In [ ]:
DATA_DIR = "data/data"
OUTPUT_PATH_A = "submission_A.csv"
OUTPUT_PATH = "submission_blend.csv"
OUTPUT_PATH_RANKAVG = "submission_blend_rankavg.csv"

RUN_VALIDATION = True
SEEDS = [0, 1, 2]

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost
import xgboost as xgb
import sklearn
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
DATA = os.path.join(DATA_DIR, "")
print("pandas", pd.__version__, "| lightgbm", lgb.__version__,
      "| catboost", catboost.__version__, "| xgboost", xgb.__version__, "| sklearn", sklearn.__version__)

In [ ]:
CUTOFF = 60


PRES_ORDER = {"2013B": 0, "2013J": 1, "2014B": 2, "2014J": 3}


CAT_COLS = ["code_module", "gender", "region", "highest_education", "imd_band",
            "age_band", "disability"]


def load():
    t = {n: pd.read_parquet(f"{DATA}{n}.parquet") for n in
         ["train", "test", "courses", "assessments", "student_assessment", "vle", "student_vle"]}
    return t


def base_features(t):
    tr, te = t["train"].copy(), t["test"].copy()
    te["target"] = np.nan
    df = pd.concat([tr, te], ignore_index=True)
    df = df.merge(t["courses"], on=["code_module", "code_presentation"], how="left")
    df["pres_idx"] = df.code_presentation.map(PRES_ORDER)
    df["is_J"] = df.code_presentation.str.endswith("J").astype(int)
    df["imd_num"] = df.imd_band.str.extract(r"^(\d+)").astype(float)
    df["age_num"] = df.age_band.map({"0-35": 0, "35-55": 1, "55<=": 2})
    df["edu_num"] = df.highest_education.map({
        "No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
        "HE Qualification": 3, "Post Graduate Qualification": 4})
    df["reg_late"] = df.date_registration.clip(lower=-400)
    return df


t = load()
for k, v in t.items():
    print(f"{k:20s} {v.shape}")

In [ ]:
def history_features(df):
    g = df[["enrollment_id", "student_id", "pres_idx", "code_module", "target"]]
    m = g.merge(g, on="student_id", suffixes=("", "_o"))
    m = m[m.enrollment_id != m.enrollment_id_o]
    prev = m[m.pres_idx_o < m.pres_idx]
    agg = prev.groupby("enrollment_id").agg(
        hist_n=("target_o", "size"), hist_fail=("target_o", "sum"),
        hist_fail_mean=("target_o", "mean"),
        hist_last_pres=("pres_idx_o", "max"))
    same_mod = prev[prev.code_module_o == prev.code_module].groupby("enrollment_id").agg(
        hist_same_n=("target_o", "size"), hist_same_fail=("target_o", "mean"))
    conc = m[m.pres_idx_o == m.pres_idx].groupby("enrollment_id").size().rename("concurrent_n")
    df = df.join(agg, on="enrollment_id").join(same_mod, on="enrollment_id").join(conc, on="enrollment_id")
    df["hist_gap"] = df.pres_idx - df.hist_last_pres
    for c in ["hist_n", "hist_same_n", "concurrent_n"]:
        df[c] = df[c].fillna(0)
    return df

In [ ]:
def vle_features(df, t):
    sv = t["student_vle"].merge(t["vle"][["id_site", "activity_type"]], on="id_site", how="left")
    key = "enrollment_id"
    out = pd.DataFrame(index=pd.Index(df[key].unique(), name=key))

    g = sv.groupby(key)
    out["clk_total"] = g.sum_click.sum()
    out["clk_rows"] = g.size()
    out["sites_n"] = g.id_site.nunique()
    out["days_active"] = g.date.nunique()
    out["first_day"] = g.date.min()
    out["last_day"] = g.date.max()
    out["days_since_last"] = CUTOFF - out.last_day

    bins = [-100, 0, 10, 20, 30, 40, 50, 60]
    sv["win"] = pd.cut(sv.date, bins=bins, right=False, labels=False)
    w = sv.pivot_table(index=key, columns="win", values="sum_click", aggfunc="sum")
    w.columns = [f"clk_w{int(c)}" for c in w.columns]
    out = out.join(w)
    wd = sv.groupby([key, "win"]).date.nunique().unstack()
    wd.columns = [f"days_w{int(c)}" for c in wd.columns]
    out = out.join(wd)
    for d in [7, 14, 21]:
        s = sv[sv.date >= CUTOFF - d]
        out[f"clk_last{d}"] = s.groupby(key).sum_click.sum()
        out[f"days_last{d}"] = s.groupby(key).date.nunique()

    sv["week"] = (sv.date // 7).clip(lower=-1)
    wk = sv[sv.date >= 0].groupby([key, "week"]).sum_click.sum().unstack(fill_value=0)
    out["weeks_active"] = (wk > 0).sum(axis=1)
    wkl = np.log1p(wk.values)
    x = np.arange(wkl.shape[1]) - (wkl.shape[1] - 1) / 2
    out.loc[wk.index, "week_slope"] = (wkl * x).sum(1) / (x ** 2).sum()
    days = sv[sv.date >= 0][[key, "date"]].drop_duplicates().sort_values([key, "date"])
    days["gap"] = days.groupby(key).date.diff()
    out["max_gap"] = days.groupby(key).gap.max()
    out["mean_gap"] = days.groupby(key).gap.mean()

    top_types = ["oucontent", "forumng", "homepage", "subpage", "resource", "url", "quiz",
                 "ouwiki", "oucollaborate", "page", "externalquiz", "questionnaire"]
    s = sv[sv.activity_type.isin(top_types)]
    at = s.pivot_table(index=key, columns="activity_type", values="sum_click", aggfunc="sum")
    at.columns = [f"clk_{c}" for c in at.columns]
    out = out.join(at)
    s2 = s[s.date >= CUTOFF - 21]
    at2 = s2.pivot_table(index=key, columns="activity_type", values="sum_click", aggfunc="sum")
    at2.columns = [f"clk21_{c}" for c in at2.columns]
    out = out.join(at2[[c for c in at2.columns if any(k in c for k in ["oucontent", "forumng", "homepage", "quiz", "subpage"])]])

    vle = t["vle"]
    n_sites = vle.groupby(["code_module", "code_presentation"]).size().rename("course_sites")
    out = out.reset_index().merge(df[[key, "code_module", "code_presentation"]], on=key, how="left")
    out = out.merge(n_sites, on=["code_module", "code_presentation"], how="left")
    out["sites_frac"] = out.sites_n / out.course_sites
    out = out.drop(columns=["code_module", "code_presentation"]).set_index(key)

    cnt_cols = [c for c in out.columns if c.startswith(("clk", "days_", "sites_n", "weeks_active"))]
    out[cnt_cols] = out[cnt_cols].fillna(0)
    out["has_vle"] = (out.clk_total > 0).astype(int)
    out["clk_per_day"] = out.clk_total / out.days_active.replace(0, np.nan)
    out["last_vs_first"] = np.log1p(out.get("clk_w6", 0)) - np.log1p(out.get("clk_w1", 0))
    return df.join(out, on=key)

In [ ]:
def assessment_features(df, t):
    a = t["assessments"]
    sa = t["student_assessment"].merge(a, on="id_assessment", how="left")
    key = "enrollment_id"
    sa["late"] = sa.date_submitted - sa.date
    sa["wscore"] = sa.score * sa.weight

    g = sa.groupby(key)
    out = pd.DataFrame({
        "as_n": g.size(),
        "as_score_mean": g.score.mean(),
        "as_score_min": g.score.min(),
        "as_score_std": g.score.std(),
        "as_late_mean": g.late.mean(),
        "as_late_max": g.late.max(),
        "as_banked": g.is_banked.sum(),
        "as_last_sub": g.date_submitted.max(),
        "as_first_sub": g.date_submitted.min(),
        "as_wscore": g.wscore.sum(),
        "as_weight": g.weight.sum(),
        "as_fail_n": g.score.apply(lambda s: (s < 40).sum()),
    })
    for typ in ["TMA", "CMA"]:
        s = sa[sa.assessment_type == typ].groupby(key)
        out[f"{typ}_n"] = s.size()
        out[f"{typ}_score"] = s.score.mean()
    first = a[a.assessment_type != "Exam"].sort_values("date").groupby(
        ["code_module", "code_presentation"]).id_assessment.first()
    f = sa[sa.id_assessment.isin(first.values)].groupby(key)
    out["first_as_score"] = f.score.first()
    out["first_as_late"] = f.late.first()

    df = df.join(out, on=key)

    due = a[(a.date < CUTOFF) & (a.assessment_type != "Exam")]
    dg = due.groupby(["code_module", "code_presentation"]).agg(
        due_n=("id_assessment", "size"), due_weight=("weight", "sum"),
        due_tma=("assessment_type", lambda s: (s == "TMA").sum()))
    df = df.merge(dg, on=["code_module", "code_presentation"], how="left")
    for c in ["due_n", "due_weight", "due_tma"]:
        df[c] = df[c].fillna(0)
    sd = sa[sa.id_assessment.isin(due.id_assessment)].groupby(key).agg(
        due_sub_n=("id_assessment", "nunique"), due_sub_score=("score", "mean"))
    df = df.join(sd, on=key)
    df["due_sub_n"] = df.due_sub_n.fillna(0)
    df["due_missed"] = df.due_n - df.due_sub_n
    df["due_frac"] = df.due_sub_n / df.due_n.replace(0, np.nan)
    nxt = a[(a.date >= CUTOFF) & (a.assessment_type != "Exam")].sort_values("date").groupby(
        ["code_module", "code_presentation"]).agg(next_due=("date", "first"))
    df = df.merge(nxt, on=["code_module", "code_presentation"], how="left")
    df["days_to_next"] = df.next_due - CUTOFF
    for c in ["as_n", "as_banked", "as_wscore", "as_weight", "as_fail_n", "TMA_n", "CMA_n"]:
        df[c] = df[c].fillna(0)
    df["as_days_since_last"] = CUTOFF - df.as_last_sub
    return df


def ordinal_assessment_features(df, t):
    key = "enrollment_id"
    pkey = ["code_module", "code_presentation"]
    due = t["assessments"]
    due = due[(due.date < CUTOFF) & (due.assessment_type != "Exam")].copy()
    due = due.sort_values(pkey + ["date"])
    due["assessment_ord"] = due.groupby(pkey).cumcount() + 1

    sa = t["student_assessment"].merge(
        due[["id_assessment", "assessment_ord", "date"]],
        on="id_assessment", how="inner",
    )
    sa["late"] = sa.date_submitted - sa.date
    sa["rank"] = sa.groupby("id_assessment").score.rank(pct=True)

    for metric in ["score", "late", "rank"]:
        values = sa.pivot_table(
            index=key, columns="assessment_ord", values=metric, aggfunc="first"
        )
        values.columns = [f"ord{int(n)}_{metric}" for n in values.columns]
        df = df.join(values, on=key)

    submitted = sa.assign(submitted=1).pivot_table(
        index=key, columns="assessment_ord", values="submitted", aggfunc="max"
    )
    submitted.columns = [f"ord{int(n)}_submitted" for n in submitted.columns]
    df = df.join(submitted, on=key)

    for ordinal in range(1, int(due.groupby(pkey).size().max()) + 1):
        column = f"ord{ordinal}_submitted"
        if column in df:
            mask = df.due_n >= ordinal
            df.loc[mask, column] = df.loc[mask, column].fillna(0)
    return df


def extra_features(df, t):
    key = "enrollment_id"
    pkey = ["code_module", "code_presentation"]
    a = t["assessments"]
    sa = t["student_assessment"].merge(a, on="id_assessment", how="left")
    due = a[(a.date < CUTOFF) & (a.assessment_type != "Exam")].copy()

    sa["score_rank"] = sa.groupby("id_assessment").score.rank(pct=True)
    sa["late_rank"] = sa.groupby("id_assessment").date_submitted.rank(pct=True)
    g = sa.groupby(key)
    out = pd.DataFrame({"as_rank_mean": g.score_rank.mean(), "as_rank_min": g.score_rank.min(),
                        "as_late_rank": g.late_rank.mean()})

    last_due = due.sort_values("date").groupby(pkey).tail(1)[pkey + ["id_assessment", "date"]]
    ld = df[[key] + pkey].merge(last_due, on=pkey, how="inner")
    ld = ld.merge(sa[[key, "id_assessment", "score", "score_rank", "date_submitted"]],
                  on=[key, "id_assessment"], how="left")
    ld = ld.set_index(key)
    out = out.join(pd.DataFrame({
        "lastdue_sub": ld.score.notna().astype(float) + ld.date_submitted.notna().astype(float),
        "lastdue_rank": ld.score_rank,
        "lastdue_late": ld.date_submitted - ld.date,
        "lastdue_age": CUTOFF - ld.date,
    }), how="outer")

    dd = df[[key] + pkey].merge(due[pkey + ["id_assessment", "weight"]], on=pkey)
    dd = dd.merge(sa[[key, "id_assessment", "score"]], on=[key, "id_assessment"], how="left")
    dd["s0"] = dd.score.fillna(0)
    dd["w"] = dd.weight + 1
    dd["ws"] = dd.s0 * dd.w
    gg = dd.groupby(key)
    out["due_score0"] = gg.s0.mean()
    out["due_wscore0"] = gg.ws.sum() / gg.w.sum()

    df = df.join(out, on=key)

    sv = t["student_vle"]
    vle = t["vle"]
    planned = vle[vle.week_from.notna() & (vle.week_from <= 8)]
    n_pl = planned.groupby(pkey).size().rename("planned_n")
    vis = sv[sv.id_site.isin(planned.id_site)].groupby(key).id_site.nunique().rename("planned_vis")
    df = df.merge(n_pl, on=pkey, how="left").join(vis, on=key)
    df["planned_frac"] = df.planned_vis.fillna(0) / df.planned_n

    sv2 = sv[sv.date >= 0].assign(week=lambda x: x.date // 7)
    wk = sv2.groupby([key, "week"]).sum_click.sum().unstack(fill_value=0)
    wk = wk.reindex(df[key]).fillna(0)
    wk.columns = [f"wk{int(c)}" for c in wk.columns]
    wk = pd.concat([df[[key] + pkey].set_index(key), wk], axis=1)
    rk = wk.groupby(pkey)[[c for c in wk.columns if c.startswith("wk")]].rank(pct=True)
    rk.columns = [f"r_{c}" for c in rk.columns]
    df = df.join(rk, on=key)
    df["r_wk_last2"] = rk[["r_wk7", "r_wk8"]].mean(axis=1).reindex(df[key]).values
    df["r_wk_drop"] = df.r_wk_last2 - rk[["r_wk0", "r_wk1", "r_wk2"]].mean(axis=1).reindex(df[key]).values
    return df

In [ ]:
WINDOWS = [(0, 7), (8, 14), (15, 30), (31, 45), (46, 59)]


TYPE_GROUPS = {
    "content": ["oucontent", "subpage", "page", "dualpane", "htmlactivity", "sharedsubpage"],
    "resource": ["resource", "url", "folder", "dataplus", "glossary"],
    "forum": ["forumng", "ouwiki", "oucollaborate", "ouelluminate"],
    "quiz": ["quiz", "externalquiz", "questionnaire", "repeatactivity"],
    "home": ["homepage"],
}


def window_features(df, t):
    key = "enrollment_id"
    pkey = ["code_module", "code_presentation"]
    grp_of = {a: g for g, lst in TYPE_GROUPS.items() for a in lst}
    sv = t["student_vle"].merge(t["vle"][["id_site", "activity_type"]], on="id_site", how="left")
    sv["tgroup"] = sv.activity_type.map(grp_of).fillna("other")
    sv = sv[sv.date >= 0]
    sv["win"] = -1
    for i, (lo, hi) in enumerate(WINDOWS):
        sv.loc[sv.date.between(lo, hi), "win"] = i
    lens = np.array([hi - lo + 1 for lo, hi in WINDOWS], dtype=float)
    idx = pd.Index(df[key], name=key)
    parts = []

    g = sv.groupby([key, "win"])
    clk = g.sum_click.sum().unstack().reindex(idx).fillna(0)
    days = g.date.nunique().unstack().reindex(idx).fillna(0)
    sites = g.id_site.nunique().unstack().reindex(idx).fillna(0)
    nw = len(WINDOWS)
    names = [f"{lo}_{hi + 1}" for lo, hi in WINDOWS]
    clk, days, sites = [x.reindex(columns=range(nw), fill_value=0) for x in (clk, days, sites)]
    wf = pd.DataFrame(index=idx)
    for i, n in enumerate(names):
        wf[f"win{n}_clk"] = clk[i]
        wf[f"win{n}_clk_per_day"] = clk[i] / lens[i]
        wf[f"win{n}_days_frac"] = days[i] / lens[i]
        wf[f"win{n}_sites"] = sites[i]
        wf[f"win{n}_intensity"] = clk[i] / days[i].replace(0, np.nan)
    lc = np.log1p(clk.values / lens)
    for i in range(1, nw):
        wf[f"trend_{names[i - 1]}_to_{names[i]}"] = lc[:, i] - lc[:, i - 1]
    x = np.arange(nw) - (nw - 1) / 2
    wf["win_clk_slope"] = (lc * x).sum(1) / (x ** 2).sum()
    df_ = days.values / lens
    wf["win_days_slope"] = (df_ * x).sum(1) / (x ** 2).sum()
    wf["win_last_vs_mean"] = lc[:, -1] - lc[:, :-1].mean(1)
    wf["win_last2_vs_first2"] = lc[:, -2:].mean(1) - lc[:, :2].mean(1)
    wf["win_active_n"] = (clk.values > 0).sum(1)
    wf["win_first_inactive"] = np.where((clk.values == 0).any(1), (clk.values == 0).argmax(1), nw)
    parts.append(wf)

    tg = sv.groupby([key, "tgroup", "win"]).sum_click.sum().unstack(["tgroup", "win"])
    tg = tg.reindex(idx).fillna(0)
    tf = pd.DataFrame(index=idx)
    total = clk.sum(1).replace(0, np.nan)
    for grp in list(TYPE_GROUPS) + ["other"]:
        cols = [c for c in tg.columns if c[0] == grp]
        tot = tg[cols].sum(1) if cols else pd.Series(0.0, index=idx)
        tf[f"type_{grp}_share"] = tot / total
        for i, n in enumerate(names):
            v = tg[(grp, i)] if (grp, i) in tg.columns else 0.0
            tf[f"win{n}_{grp}_clk"] = v
        first = tg[(grp, 0)] if (grp, 0) in tg.columns else 0.0
        last = tg[(grp, nw - 1)] if (grp, nw - 1) in tg.columns else 0.0
        tf[f"type_{grp}_trend"] = np.log1p(last / lens[-1]) - np.log1p(first / lens[0])
    ts = sv.groupby([key, "tgroup"]).agg(d=("date", "nunique"), s=("id_site", "nunique")).unstack()
    ts = ts.reindex(idx)
    for grp in list(TYPE_GROUPS) + ["other"]:
        tf[f"type_{grp}_days"] = ts[("d", grp)].fillna(0) if ("d", grp) in ts.columns else 0
        tf[f"type_{grp}_sites"] = ts[("s", grp)].fillna(0) if ("s", grp) in ts.columns else 0
    vle = t["vle"].assign(tgroup=lambda v: v.activity_type.map(grp_of).fillna("other"))
    n_type = vle.groupby(pkey + ["tgroup"]).size().unstack(fill_value=0)
    n_type = df[pkey].merge(n_type, left_on=pkey, right_index=True, how="left").set_index(idx)
    for grp in list(TYPE_GROUPS):
        tf[f"type_{grp}_coverage"] = tf[f"type_{grp}_sites"] / n_type[grp].replace(0, np.nan)
    tf["type_diversity"] = sv.groupby(key).activity_type.nunique().reindex(idx).fillna(0)
    parts.append(tf)

    act = np.zeros((len(idx), CUTOFF), dtype=bool)
    pos = pd.Series(np.arange(len(idx)), index=idx)
    d = sv[[key, "date"]].drop_duplicates()
    act[pos.reindex(d[key]).values, d.date.values.astype(int)] = True
    inact = ~act
    nf = pd.DataFrame(index=idx)
    nf["inactive_days"] = inact.sum(1)
    run = np.zeros(len(idx)); longest = np.zeros(len(idx)); n7 = np.zeros(len(idx))
    for day in range(CUTOFF):
        run = np.where(inact[:, day], run + 1, 0)
        longest = np.maximum(longest, run)
        n7 += (run == 7)
    nf["inactive_longest"] = longest
    nf["inactive_current"] = run
    nf["inactive_breaks7"] = n7
    for i, (lo, hi) in enumerate(WINDOWS):
        seg = inact[:, lo:hi + 1]
        r = np.zeros(len(idx)); m = np.zeros(len(idx))
        for j in range(seg.shape[1]):
            r = np.where(seg[:, j], r + 1, 0); m = np.maximum(m, r)
        nf[f"win{names[i]}_longest_inactive"] = m
    wk_act = act[:, :56].reshape(len(idx), 8, 7).any(2)
    tail = np.zeros(len(idx))
    alive = np.ones(len(idx), dtype=bool)
    for w in range(7, -1, -1):
        alive &= wk_act[:, w]; tail += alive
    nf["active_weeks_tail"] = tail
    nf["active_weeks_n"] = wk_act.sum(1)
    parts.append(nf)

    a = t["assessments"]
    due = a[(a.date < CUTOFF) & (a.assessment_type != "Exam")].copy()
    due["win"] = -1
    for i, (lo, hi) in enumerate(WINDOWS):
        due.loc[due.date.between(lo, hi), "win"] = i
    sa = t["student_assessment"]
    dd = df[[key] + pkey].merge(due[pkey + ["id_assessment", "assessment_type", "weight", "win", "date"]], on=pkey)
    dd = dd.merge(sa[[key, "id_assessment", "date_submitted", "score"]], on=[key, "id_assessment"], how="left")
    dd["sub"] = dd.date_submitted.notna().astype(float)
    dd["ontime"] = (dd.date_submitted <= dd.date).astype(float)
    dd["passed"] = (dd.score >= 40).astype(float)
    af = pd.DataFrame(index=idx)
    gdd = dd.groupby(key)
    af["done_frac"] = gdd["sub"].mean()
    af["ontime_frac"] = gdd.ontime.mean()
    af["passed_frac"] = gdd.passed.mean()
    dd["w"] = dd.weight + 1
    af["done_wfrac"] = (dd["sub"] * dd.w).groupby(dd[key]).sum() / gdd.w.sum()
    for typ in ["TMA", "CMA"]:
        s = dd[dd.assessment_type == typ].groupby(key)["sub"]
        af[f"done_frac_{typ}"] = s.mean()
    wsub = dd.groupby([key, "win"])["sub"].mean().unstack()
    for i, n in enumerate(names):
        af[f"win{n}_done_frac"] = wsub[i] if i in wsub.columns else np.nan
    early = sa.merge(a[a.date >= CUTOFF][["id_assessment"]], on="id_assessment")
    af["early_future_sub"] = early.groupby(key).size().reindex(idx).fillna(0)
    parts.append(af.reindex(idx))

    feats = pd.concat(parts, axis=1)
    rel_cols = [f"win{n}_clk" for n in names] + [f"win{n}_days_frac" for n in names] + \
               ["win_clk_slope", "inactive_days", "inactive_longest", "done_wfrac"] + \
               [f"type_{g}_coverage" for g in TYPE_GROUPS]
    r = pd.concat([df[pkey].set_index(idx), feats[rel_cols]], axis=1).groupby(pkey)[rel_cols].rank(pct=True)
    feats = pd.concat([feats, r.add_prefix("r_")], axis=1)
    return pd.concat([df.reset_index(drop=True), feats.reset_index(drop=True)], axis=1)

In [ ]:
def _entropy(counts):
    p = counts / counts.sum(1, keepdims=True).clip(min=1e-9)
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -(np.where(p > 0, p * np.log(p), 0)).sum(1)
    return np.where(counts.sum(1) > 0, h, np.nan)


def idea_features(df, t):
    key = "enrollment_id"
    pkey = ["code_module", "code_presentation"]
    idx = pd.Index(df[key], name=key)
    grp_of = {a: g for g, lst in TYPE_GROUPS.items() for a in lst}
    vle = t["vle"].assign(tgroup=lambda v: v.activity_type.map(grp_of).fillna("other"))
    sv = t["student_vle"].merge(vle[["id_site", "activity_type", "tgroup", "week_from"]], on="id_site", how="left")
    sv = sv[sv.date >= 0]
    out = pd.DataFrame(index=idx)

    for lo, hi in WINDOWS:
        c = sv[sv.date.between(lo, hi)].groupby(key).sum_click.sum().reindex(idx).fillna(0)
        tmp = pd.concat([df[pkey].set_index(idx), c.rename("c")], axis=1)
        med = tmp.groupby(pkey).c.transform("median")
        mean = tmp.groupby(pkey).c.transform("mean")
        out[f"rel_med_{lo}_{hi + 1}"] = np.log1p(c) - np.log1p(med)
        out[f"rel_mean_{lo}_{hi + 1}"] = c / mean.replace(0, np.nan)

    wk = sv[sv.date < 56].assign(w=lambda x: x.date // 7).groupby([key, "w"]).sum_click.sum()
    wk = np.log1p(wk.unstack().reindex(idx).reindex(columns=range(8)).fillna(0).values)
    d1, d2 = np.diff(wk, axis=1), np.diff(wk, n=2, axis=1)
    out["wk_d1_last"] = d1[:, -1]
    out["wk_d1_min"] = d1.min(1)
    out["wk_d1_mean_last3"] = d1[:, -3:].mean(1)
    out["wk_d2_last"] = d2[:, -1]
    out["wk_d2_min"] = d2.min(1)
    out["wk_d2_mean_last3"] = d2[:, -3:].mean(1)
    out["wk_peak_week"] = wk.argmax(1)
    out["wk_from_peak"] = wk[:, -1] - wk.max(1)

    typ = sv.groupby([key, "activity_type"]).sum_click.sum().unstack().reindex(idx).fillna(0).values
    out["entropy_types"] = _entropy(typ)
    site_p = sv.groupby([key, "id_site"]).sum_click.sum()
    s = site_p / site_p.groupby(level=0).transform("sum")
    out["entropy_sites"] = (-(s * np.log(s)).groupby(level=0).sum()).reindex(idx)
    day = sv.groupby([key, "date"]).sum_click.sum().unstack().reindex(idx).fillna(0).values
    out["entropy_days"] = _entropy(day)
    out["entropy_days_norm"] = out.entropy_days / np.log(CUTOFF)

    groups = list(TYPE_GROUPS) + ["other"]
    def tdist(lo, hi):
        m = sv[sv.date.between(lo, hi)].groupby([key, "tgroup"]).sum_click.sum().unstack()
        return m.reindex(idx).reindex(columns=groups).fillna(0).values
    p1, p2 = tdist(0, 29), tdist(30, 59)
    q1 = p1 / p1.sum(1, keepdims=True).clip(min=1e-9)
    q2 = p2 / p2.sum(1, keepdims=True).clip(min=1e-9)
    mid = (q1 + q2) / 2
    def kl(a, b):
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(a > 0, a * np.log(a / np.where(b > 0, b, 1)), 0).sum(1)
    js = 0.5 * kl(q1, mid) + 0.5 * kl(q2, mid)
    out["type_shift_js"] = np.where((p1.sum(1) > 0) & (p2.sum(1) > 0), js, np.nan)
    for i, g in enumerate(groups[:4]):
        out[f"type_shift_{g}"] = q2[:, i] - q1[:, i]

    c1 = sv[sv.date < 30].groupby(key).sum_click.sum().reindex(idx).fillna(0)
    c2 = sv[sv.date >= 30].groupby(key).sum_click.sum().reindex(idx).fillna(0)
    a1 = sv[sv.date < 30].groupby(key).date.nunique().reindex(idx).fillna(0)
    a2 = sv[sv.date >= 30].groupby(key).date.nunique().reindex(idx).fillna(0)
    out["D_clicks"] = np.log((c1 + 1) / (c2 + 1))
    out["D_days"] = np.log((a1 + 1) / (a2 + 1))
    out["D_clicks_abs_diff"] = c2 - c1
    out["D_days_abs_diff"] = a2 - a1
    out["half1_days"], out["half2_days"] = a1, a2

    sa = t["student_assessment"].merge(t["assessments"], on="id_assessment", how="left")
    ms = sa.groupby(key).score.mean().reindex(idx)
    tot = sv.groupby(key).sum_click.sum().reindex(idx).fillna(0)
    out["E_eff"] = ms / np.log1p(tot).replace(0, np.nan)
    tmp = pd.concat([df[pkey].set_index(idx), out[["E_eff"]], ms.rename("ms"), tot.rename("tot")], axis=1)
    rs = tmp.groupby(pkey).ms.rank(pct=True)
    rc = tmp.groupby(pkey).tot.rank(pct=True)
    out["E_rank_diff"] = rs - rc
    out["r_E_eff"] = tmp.groupby(pkey).E_eff.rank(pct=True)

    sa["late"] = sa.date_submitted - sa.date
    sa = sa.sort_values([key, "date"])
    sa["k"] = sa.groupby(key).cumcount()
    def slope(g):
        if len(g) < 2:
            return np.nan
        x = g.k - g.k.mean()
        return (x * (g.late - g.late.mean())).sum() / (x ** 2).sum()
    out["late_slope"] = sa.groupby(key)[["k", "late"]].apply(slope).reindex(idx)
    out["late_last"] = sa.groupby(key).late.last().reindex(idx)
    out["late_frac"] = sa.assign(l=sa.late > 0).groupby(key).l.mean().reindex(idx)

    vle["avail"] = vle.week_from.isna() | (vle.week_from <= 8)
    vle["current"] = vle.week_from.between(5, 8)
    av = vle[vle.avail].groupby(pkey).size().rename("avail_n")
    cu = vle[vle.current].groupby(pkey).size().rename("cur_n")
    seen = sv[["id_site", key]].drop_duplicates().merge(vle[["id_site", "avail", "current", "week_from"]], on="id_site")
    plan = pd.DataFrame(index=idx)
    plan["avail_seen"] = seen[seen.avail].groupby(key).size().reindex(idx).fillna(0)
    plan["cur_seen"] = seen[seen.current].groupby(key).size().reindex(idx).fillna(0)
    plan["ahead_seen"] = seen[seen.week_from > 8].groupby(key).size().reindex(idx).fillna(0)
    plan = pd.concat([df[pkey].set_index(idx), plan], axis=1)
    plan = plan.merge(av, left_on=pkey, right_index=True, how="left").merge(cu, left_on=pkey, right_index=True, how="left")
    out["plan_avail_frac"] = plan.avail_seen / plan.avail_n.replace(0, np.nan)
    out["plan_current_frac"] = plan.cur_seen / plan.cur_n.replace(0, np.nan)
    out["plan_ahead_n"] = plan.ahead_seen
    first = sv.groupby([key, "id_site"]).date.min().reset_index().merge(vle[["id_site", "week_from"]], on="id_site")
    first = first[first.week_from.notna()]
    first["lag"] = first.date / 7 - first.week_from
    out["plan_open_lag_mean"] = first.groupby(key).lag.mean().reindex(idx)
    out["plan_open_ontime"] = first.assign(o=first.lag <= 1).groupby(key).o.mean().reindex(idx)

    r_cols = ["plan_avail_frac", "plan_current_frac", "entropy_types", "D_clicks", "wk_d1_last"]
    r = pd.concat([df[pkey].set_index(idx), out[r_cols]], axis=1).groupby(pkey)[r_cols].rank(pct=True)
    out = pd.concat([out, r.add_prefix("r_")], axis=1)
    return pd.concat([df.reset_index(drop=True), out.reset_index(drop=True)], axis=1)


def cluster_features(df, k=6, seed=0):
    from sklearn.cluster import KMeans
    cols = [f"r_wk{i}" for i in range(9)] + ["done_frac", "r_as_score_mean", "inactive_longest"]
    X = df[cols].copy()
    X["inactive_longest"] = X.inactive_longest / CUTOFF
    X = X.fillna({"done_frac": 1, "r_as_score_mean": 0}).fillna(0).values
    km = KMeans(k, n_init=10, random_state=seed).fit(X)
    d = km.transform(X)
    out = pd.DataFrame(d, columns=[f"clu_dist{i}" for i in range(k)])
    out["clu_id"] = km.labels_
    return pd.concat([df.reset_index(drop=True), out], axis=1)

In [ ]:
def deadline_features(df, t):
    key = "enrollment_id"
    pkey = ["code_module", "code_presentation"]
    idx = pd.Index(df[key], name=key)
    a = t["assessments"]
    due = a[(a.date < CUTOFF) & (a.date >= 3) & (a.assessment_type != "Exam")]
    daily = t["student_vle"].groupby([key, "date"]).sum_click.sum().unstack().reindex(idx)
    D = daily.reindex(columns=range(-30, CUTOFF)).fillna(0).values
    col0 = 30
    end = CUTOFF + col0
    base = D[:, col0:].mean(1) + 0.1
    cs = np.concatenate([np.zeros((D.shape[0], 1)), D.cumsum(1)], 1)

    def win_sum(r, lo, hi):
        return cs[r, np.clip(hi, 0, end)] - cs[r, np.clip(lo, 0, end)]

    pairs = df[[key] + pkey].reset_index(drop=True).merge(due[pkey + ["id_assessment", "date"]], on=pkey)
    pairs = pairs.merge(t["student_assessment"][[key, "id_assessment", "date_submitted"]],
                        on=[key, "id_assessment"], how="left")
    row = pd.Series(np.arange(len(idx)), index=idx).reindex(pairs[key]).values
    dl = pairs.date.values.astype(int) + col0
    b = base[row]
    pairs["pre7"] = win_sum(row, dl - 7, dl) / 7 / b
    pairs["pre3"] = win_sum(row, dl - 3, dl) / 3 / b
    post_len = np.clip(np.minimum(7, end - dl - 1), 1, None)
    pairs["post7"] = win_sum(row, dl + 1, dl + 1 + post_len) / post_len / b
    pairs["pre_active"] = (win_sum(row, dl - 7, dl) > 0).astype(float)
    sub = pairs.date_submitted.values
    has = ~np.isnan(sub)
    ds = np.where(has, sub, 0).astype(int) + col0
    alen = np.clip(np.minimum(7, end - ds - 1), 1, None)
    pairs["after_sub7"] = np.where(has, win_sum(row, ds + 1, ds + 1 + alen) / alen / b, np.nan)
    pairs["missed_but_active"] = (~has & (pairs.pre7 > 0.5)).astype(float)
    pairs["missed_inactive"] = (~has & (pairs.pre_active == 0)).astype(float)
    g = pairs.groupby(key)
    out = pd.DataFrame({
        "dl_pre7": g.pre7.mean(), "dl_pre3": g.pre3.mean(), "dl_post7": g.post7.mean(),
        "dl_pre_active": g.pre_active.mean(), "dl_after_sub7": g.after_sub7.mean(),
        "dl_missed_but_active": g.missed_but_active.sum(), "dl_missed_inactive": g.missed_inactive.sum(),
    }).reindex(idx)
    last = pairs.sort_values("date").groupby(key).tail(1).set_index(key)
    out["dl_last_pre7"] = last.pre7.reindex(idx)
    out["dl_last_post7"] = last.post7.reindex(idx)
    out["dl_prep_ratio"] = np.log((out.dl_pre7 + 0.1) / (out.dl_post7 + 0.1))
    rc = ["dl_pre7", "dl_post7", "dl_after_sub7"]
    r = pd.concat([df[pkey].set_index(idx), out[rc]], axis=1).groupby(pkey)[rc].rank(pct=True)
    out = pd.concat([out, r.add_prefix("r_")], axis=1)
    return pd.concat([df.reset_index(drop=True), out.reset_index(drop=True)], axis=1)

In [ ]:
def relative_features(df, cols):
    ranks = df.groupby(["code_module", "code_presentation"])[cols].rank(pct=True)
    return pd.concat([df, ranks.add_prefix("r_")], axis=1)


REL_COLS = ["clk_total", "days_active", "clk_last7", "clk_last14", "clk_last21", "days_last14",
           "clk_w5", "clk_w6", "sites_frac", "weeks_active", "as_score_mean", "as_wscore",
           "due_sub_score", "last_day", "clk_oucontent", "clk_forumng", "clk_quiz",
           "date_registration", "week_slope", "max_gap"]


def build(t):
    df = base_features(t)
    df = history_features(df)
    df = vle_features(df, t)
    df = assessment_features(df, t)
    df = ordinal_assessment_features(df, t)
    df = df.copy()
    df = extra_features(df, t)
    base_cols = list(df.columns)
    df = window_features(df, t)
    df = relative_features(df, REL_COLS)
    df = idea_features(df, t)
    df = cluster_features(df)
    df = deadline_features(df, t)
    for c in CAT_COLS:
        df[c] = df[c].astype("category")
    return df, base_cols


df, base_cols = build(t)
DROP = ["enrollment_id", "student_id", "code_presentation", "target", "pres_idx", "hist_last_pres",
        "course_sites", "module_presentation_length", "next_due", "days_to_next", "due_weight", "planned_n"]
FEATS = [c for c in df.columns if c not in DROP]
CATS = [c for c in FEATS if str(df[c].dtype) == "category"]
tr, te = df[df.target.notna()], df[df.target.isna()]
B_DROP = {"enrollment_id", "student_id", "code_presentation", "target", "pres_idx", "hist_last_pres"}
FEATS_B = [c for c in base_cols if c not in B_DROP] + ["r_" + c for c in REL_COLS]
print("признаков A:", len(FEATS), "| признаков B:", len(FEATS_B), "| train:", tr.shape[0], "| test:", te.shape[0])

In [ ]:
LGB_PARAMS = dict(objective="binary", learning_rate=0.02, num_leaves=15, min_child_samples=100,
                  feature_fraction=0.3, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5,
                  deterministic=True, force_col_wise=True, verbose=-1)


def cb_x(d):
    return d[FEATS].assign(**{c: d[c].astype(object).fillna("NA").astype(str) for c in CATS})


def fit_predict(a, b, seeds=SEEDS):
    pl, pc, px = [], [], []
    for s in seeds:
        m = lgb.train({**LGB_PARAMS, "seed": s}, lgb.Dataset(a[FEATS], a.target), 1000)
        pl.append(m.predict(b[FEATS]))
        c = CatBoostClassifier(iterations=1500, learning_rate=0.03, depth=6, random_seed=s,
                               verbose=0, cat_features=CATS, thread_count=-1, allow_writing_files=False)
        c.fit(cb_x(a), a.target)
        pc.append(c.predict_proba(cb_x(b))[:, 1])
        x = xgb.XGBClassifier(n_estimators=1200, learning_rate=0.02, max_depth=4, subsample=0.8,
                              colsample_bytree=0.3, min_child_weight=10, reg_lambda=5,
                              enable_categorical=True, tree_method="hist", random_state=s)
        x.fit(a[FEATS], a.target)
        px.append(x.predict_proba(b[FEATS])[:, 1])
    preds = {"LightGBM": np.mean(pl, 0), "CatBoost": np.mean(pc, 0), "XGBoost": np.mean(px, 0)}
    blend = np.mean([rankdata(p) / len(p) for p in preds.values()], 0)
    return preds, blend, m

In [ ]:
B_PARAMS = dict(objective="binary", learning_rate=0.02, num_leaves=31, min_child_samples=50,
                feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1, verbosity=-1)
B_SECONDARY = {**B_PARAMS, "num_leaves": 15, "feature_fraction": 0.6, "lambda_l2": 5}
B_LOCAL = {**B_PARAMS, "num_leaves": 7, "min_child_samples": 25, "feature_fraction": 0.8,
           "bagging_fraction": 0.9, "lambda_l2": 10, "seed": 23}
B_ROUNDS, B_SECONDARY_ROUNDS, B_LOCAL_ROUNDS = 316, 392, 300


def rank01(p):
    return rankdata(p) / len(p)


def linear_model(a, b, feats):
    cats = [c for c in feats if str(a[c].dtype) == "category"]
    nums = [c for c in feats if c not in cats]

    def frame(d):
        x = d[feats].copy()
        for c in cats:
            x[c] = x[c].astype(object).where(x[c].notna(), "NA").astype(str)
        return x

    model = make_pipeline(
        ColumnTransformer([
            ("category", OneHotEncoder(handle_unknown="ignore"), cats),
            ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                      FunctionTransformer(lambda x: np.sign(x) * np.log1p(np.abs(x))),
                                      StandardScaler()), nums)]),
        LogisticRegression(C=0.1, max_iter=1000, solver="liblinear"))
    return model.fit(frame(a), a.target).predict_proba(frame(b))[:, 1]


def model_b(a, b):
    f = FEATS_B
    main = np.mean([lgb.train({**B_PARAMS, "seed": s}, lgb.Dataset(a[f], a.target), B_ROUNDS).predict(b[f])
                    for s in [17, 29, 42, 101, 2026]], 0)
    second = np.mean([lgb.train({**B_SECONDARY, "seed": s}, lgb.Dataset(a[f], a.target),
                                B_SECONDARY_ROUNDS).predict(b[f])
                      for s in [17, 42, 101]], 0)
    general = 0.5 * main + 0.5 * second
    local = general.copy()
    for module in b.code_module.unique():
        ma, mb = a[a.code_module == module], (b.code_module == module).values
        if len(ma):
            local[mb] = lgb.train(B_LOCAL, lgb.Dataset(ma[f], ma.target), B_LOCAL_ROUNDS).predict(b.loc[mb, f])
    trees = 0.8 * general + 0.2 * local
    return 0.8 * trees + 0.2 * rank01(linear_model(a, b, f))


def blend(pred_a, pred_b, pred_lin):
    return 0.5 * rank01(pred_a) + 0.4 * rank01(pred_b) + 0.1 * rank01(pred_lin)

In [ ]:
if RUN_VALIDATION:
    a, b = tr[tr.pres_idx < 2], tr[tr.pres_idx == 2]
    preds, pa, m_val = fit_predict(a, b)
    pb_ = model_b(a, b)
    plin = linear_model(a, b, FEATS)
    pb = blend(pa, pb_, plin)
    val = pd.Series({**{k: roc_auc_score(b.target, p) for k, p in preds.items()},
                     "A: три бустинга": roc_auc_score(b.target, pa),
                     "B: базовые признаки": roc_auc_score(b.target, pb_),
                     "Логрег на всех признаках": roc_auc_score(b.target, plin),
                     "Смесь A + B + логрег": roc_auc_score(b.target, pb)}, name="AUC 2014B").round(4)
    display(val.to_frame())
    by_module = pd.DataFrame({"module": b.code_module.astype(str), "y": b.target, "p": pb})
    display(by_module.groupby("module").apply(lambda g: pd.Series(
        {"n": len(g), "доля 1": round(g.y.mean(), 3), "AUC": round(roc_auc_score(g.y, g.p), 4)})))

In [ ]:
preds_te, pred_a, _ = fit_predict(tr, te)
pred_b = model_b(tr, te)
pred_lin = linear_model(tr, te, FEATS)
pred = blend(pred_a, pred_b, pred_lin)

sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
for path, p in [(OUTPUT_PATH_A, pred_a), (OUTPUT_PATH, pred)]:
    sub = pd.DataFrame({"id": te.enrollment_id.values, "prediction": p})
    assert len(sub) == len(sample) and set(sub.id) == set(sample.id) and sub.prediction.notna().all()
    sub.to_csv(path, index=False)
    print(path, sub.shape)
print(round(pd.Series(pred_a).corr(pd.Series(pred_b), method="spearman"), 4))
sub.head()

In [ ]:
rankavg = 0.5 * rank01(pred_a) + 0.5 * rank01(pred)
sub = pd.DataFrame({"id": te.enrollment_id.values, "prediction": rankavg})
assert len(sub) == len(sample) and set(sub.id) == set(sample.id)
sub.to_csv(OUTPUT_PATH_RANKAVG, index=False)
print(OUTPUT_PATH_RANKAVG, sub.shape)
sub.head()